In [10]:
import numpy as np
import xarray as xr
import plotly.graph_objs as go
from dataclasses import dataclass

In [11]:
# Data paths
ERA5_MAX_TEMP_GLOB = './datasets/ERA5_Land/max_temperature/*.nc'
METEO_SWISS_MAX_TEMP_GLOB = './datasets/MeteoSwiss/TmaxD_v2.0_swiss.lv95/*.nc'

# Year filter (None for all data)
DATA_YEAR_START = 2023

In [12]:
# Load ERA5 data
era5_ds = xr.open_mfdataset(ERA5_MAX_TEMP_GLOB, combine='by_coords')
if DATA_YEAR_START is not None:
    era5_ds = era5_ds.sel(time=slice(f'{DATA_YEAR_START}-01-01', None))

print("ERA5 data:")
print(era5_ds)

ERA5 data:
<xarray.Dataset> Size: 3MB
Dimensions:    (time: 365, latitude: 29, longitude: 61)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2023-01-01 2023-01-02 ... 2023-12-31
  * latitude   (latitude) float64 232B 48.2 48.1 48.0 47.9 ... 45.6 45.5 45.4
  * longitude  (longitude) float64 488B 5.0 5.1 5.2 5.3 ... 10.7 10.8 10.9 11.0
Data variables:
    t2m_max    (time, latitude, longitude) float32 3MB dask.array<chunksize=(365, 29, 61), meta=np.ndarray>


In [13]:
# Load MeteoSwiss data
meteoswiss_ds = xr.open_mfdataset(METEO_SWISS_MAX_TEMP_GLOB, combine='by_coords', data_vars='all')
if DATA_YEAR_START is not None:
    meteoswiss_ds = meteoswiss_ds.sel(time=slice(f'{DATA_YEAR_START}-01-01', None))

print("MeteoSwiss data:")
print(meteoswiss_ds)

MeteoSwiss data:
<xarray.Dataset> Size: 130MB
Dimensions:                 (time: 365, N: 240, E: 370)
Coordinates:
  * time                    (time) datetime64[ns] 3kB 2023-01-01 ... 2023-12-31
  * N                       (N) float64 2kB 1.064e+06 1.066e+06 ... 1.304e+06
  * E                       (E) float64 3kB 2.474e+06 2.476e+06 ... 2.844e+06
    lon                     (N, E) float32 355kB dask.array<chunksize=(240, 370), meta=np.ndarray>
    lat                     (N, E) float32 355kB dask.array<chunksize=(240, 370), meta=np.ndarray>
Data variables:
    swiss_lv95_coordinates  (time) float32 1kB 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0
    TmaxD                   (time, N, E) float32 130MB dask.array<chunksize=(365, 240, 370), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.8
    institution:  Data produced at Federal Office of Meteorology and Climatol...
    title:        Spatial Climate Analyses Dataset of MeteoSwiss (TmaxD)
    history:      2023-11-23 16:42:41 - generated by gr

In [14]:
# Get ERA5 coordinates (regular lat/lon grid)
era5_lon, era5_lat = np.meshgrid(era5_ds['longitude'].values, era5_ds['latitude'].values)
era5_coords = {
    'latitude': era5_lat.flatten(),
    'longitude': era5_lon.flatten()
}

print(f"ERA5 grid: {len(era5_coords['latitude'])} points")
print(f"  Lat range: [{era5_coords['latitude'].min():.4f}, {era5_coords['latitude'].max():.4f}]")
print(f"  Lon range: [{era5_coords['longitude'].min():.4f}, {era5_coords['longitude'].max():.4f}]")

ERA5 grid: 1769 points
  Lat range: [45.4000, 48.2000]
  Lon range: [5.0000, 11.0000]


In [15]:
# Get MeteoSwiss coordinates (flatten the N, E grid)
# MeteoSwiss has 'lat' and 'lon' variables in WGS84 degrees
meteoswiss_lat = meteoswiss_ds['lat'].values.flatten()
meteoswiss_lon = meteoswiss_ds['lon'].values.flatten()

# Filter out NaN coordinates if any
valid_mask = ~(np.isnan(meteoswiss_lat) | np.isnan(meteoswiss_lon))
meteoswiss_coords = {
    'latitude': meteoswiss_lat[valid_mask],
    'longitude': meteoswiss_lon[valid_mask]
}

print(f"MeteoSwiss grid: {len(meteoswiss_coords['latitude'])} points")
print(f"  Lat range: [{meteoswiss_coords['latitude'].min():.4f}, {meteoswiss_coords['latitude'].max():.4f}]")
print(f"  Lon range: [{meteoswiss_coords['longitude'].min():.4f}, {meteoswiss_coords['longitude'].max():.4f}]")

MeteoSwiss grid: 88800 points
  Lat range: [45.6886, 47.8820]
  Lon range: [5.7613, 10.6919]


In [16]:
# Create the grid visualization
fig = go.Figure()

# MeteoSwiss target points (high resolution)
fig.add_trace(go.Scattergeo(
    lon=meteoswiss_coords['longitude'],
    lat=meteoswiss_coords['latitude'],
    mode='markers',
    marker_color='olivedrab',
    marker=dict(size=2),
    name='MeteoSwiss (target)',
    opacity=0.6
))

# ERA5 input points (coarse resolution)
fig.add_trace(go.Scattergeo(
    lon=era5_coords['longitude'],
    lat=era5_coords['latitude'],
    mode='markers',
    marker_color='sandybrown',
    marker=dict(size=6),
    name='ERA5 (input)'
))

fig.update_layout(
    title='MeteoSwiss target grid and ERA5 input grid',
    geo=dict(
        scope='europe',
        fitbounds='locations',
        showland=True,
        landcolor='rgb(243, 243, 243)',
        countrycolor='rgb(204, 204, 204)',
    ),
    width=1000,
    height=700,
    legend=dict(
        yanchor='top',
        y=0.99,
        xanchor='left',
        x=0.01
    )
)

fig.show()